> **Runtime**: Google Colab with GPU runtime (A100 recommended; T4 will work but slower).
>
> **Dependencies**: TensorFlow ≥ 2.12, TensorFlow-Probability ≥ 0.20. Installed by the first cell.
>
> **Codebase**: requires the `advanced_particle_filter` codebase to be uploaded as a zip and extracted at `/content/advanced_particle_filter/`. The first setup cell handles this.
>
> **What it does**: SVSSM calibration via HMC: soft vs Sinkhorn-OT resampler comparison.

---

# SVSSM + DPF + HMC: Soft vs OT Comparison

Two experiments on the 2-asset stochastic volatility SSM:
1. **Soft resampler N=1000** — push particle count for best gradient SNR
2. **OT resampler N=80** — maximum affordable on A100 40GB

Based on linear-Gaussian diagnostics: HMC acceptance is governed by
gradient SNR, which scales with N. Soft can afford large N (O(N·D) memory);
OT is memory-constrained (O(N²·N_s) backward memory).

**Runtime:** GPU (A100 recommended). ~2 hrs soft + ~1.5 hrs OT = ~3.5 hrs total.

---
## 0. Setup

In [ ]:
!pip install -q tensorflow-probability

import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import time

print(f'TF {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
if gpus:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print('No GPU! Switch to GPU runtime.')

In [ ]:
import os, zipfile
if not os.path.isdir('/content/advanced_particle_filter'):
    from google.colab import files
    uploaded = files.upload()
    with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
        z.extractall('/content')
    print('Package extracted.')
else:
    print('Package already present.')

import sys
sys.path.insert(0, '/content')

## 1. Smoke Test

In [ ]:
from advanced_particle_filter.tf_models.svssm import simulate_svssm, SVSSMParams
from advanced_particle_filter.tf_filters import TFDifferentiableParticleFilter
from advanced_particle_filter.hmc.parameterization import unpack_batched, TOTAL_DIM

DTYPE = tf.float64

# Quick forward + gradient check
mu_test = tf.constant([-1.0, 0.5], dtype=DTYPE)
Phi_test = tf.constant([[0.85, 0.12], [0.02, 0.90]], dtype=DTYPE)
Lchol_test = tf.constant(
    np.linalg.cholesky([[0.0225, 0.018], [0.018, 0.16]]), dtype=DTYPE
)
rng = tf.random.Generator.from_seed(0)
_, y_test = simulate_svssm(mu_test, Phi_test, Lchol_test, T=50, rng=rng)

dpf_test = TFDifferentiableParticleFilter(
    n_particles=100, resampler='soft', alpha=0.5, dtype=DTYPE,
)

B = 4
params_test = SVSSMParams(
    mu=tf.tile(mu_test[tf.newaxis, :], [B, 1]),
    Phi=tf.tile(Phi_test[tf.newaxis, :, :], [B, 1, 1]),
    Sigma_eta_chol=tf.tile(Lchol_test[tf.newaxis, :, :], [B, 1, 1]),
)
pf_rng = tf.random.Generator.from_seed(1)

# Forward
t0 = time.time()
result = dpf_test.filter(params_test, y_test, pf_rng)
print(f'Forward (first trace): {time.time()-t0:.2f}s')
print(f'  log_evidence = {result.log_evidence.numpy().round(2)}')

t0 = time.time()
result2 = dpf_test.filter(params_test, y_test, pf_rng)
print(f'Forward (cached): {time.time()-t0:.3f}s')

# Gradient
z = tf.Variable(tf.zeros([4, TOTAL_DIM], dtype=DTYPE) + 0.1 * tf.random.normal([4, TOTAL_DIM], dtype=DTYPE))
with tf.GradientTape() as tape:
    params_z = unpack_batched(z)
    res = dpf_test.filter(params_z, y_test, pf_rng)
    loss = tf.reduce_sum(res.log_evidence)
grad = tape.gradient(loss, z)
print(f'Gradient norms: {tf.norm(grad, axis=-1).numpy().round(2)}')
print(f'All finite: {bool(tf.reduce_all(tf.math.is_finite(grad)).numpy())}')
print('\nSmoke test passed.')

---
## 2. Experiment 1: Soft Resampler N=1000

Our best shot at convergence. Large N for high gradient SNR.

**Memory:** O(T×B×N×D) = 150×4×1000×2×8 ≈ 10 MB backward. Trivial.

**Runtime estimate (A100):** ~2 hours.

In [ ]:
from advanced_particle_filter.hmc.run_hmc_poc import main

out_soft = main(
    T=150,
    B_chain=4,
    n_mc=1,
    n_particles=1000,
    resampler='soft',
    alpha=0.5,
    num_burnin=300,
    num_samples=300,
    num_leapfrog=15,
    initial_step_size=0.02,
    jitter=0.02,
    seed=42,
)

In [ ]:
# Trace plots for soft
import matplotlib.pyplot as plt

samples_z = out_soft['samples_z']
param_names = ['mu_0', 'log(gap)',
               'Phi_00', 'Phi_01', 'Phi_10', 'Phi_11',
               'log_L11', 'log_L22', 'L21']

fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
for i, ax in enumerate(axes.flat):
    for b in range(samples_z.shape[1]):
        ax.plot(samples_z[:, b, i], alpha=0.7, lw=0.8, label=f'chain {b}')
    ax.set_title(param_names[i])
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(loc='upper right', fontsize=8)
plt.suptitle('Soft N=1000 Trace Plots', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Experiment 2: OT Resampler N=80

Maximum particle count on A100 40GB with OT backward memory.

**Memory:** O(T×N_s×B×N²) = 150×100×4×80²×8 ≈ 31 GB backward.

**Runtime estimate (A100):** ~1.5 hours.

If you have A100 80GB, try N=100 instead (48 GB backward memory).

In [ ]:
out_ot = main(
    T=150,
    B_chain=4,
    n_mc=1,
    n_particles=80,
    resampler='sinkhorn',
    epsilon=0.1,
    sinkhorn_iters=100,
    num_burnin=200,
    num_samples=200,
    num_leapfrog=5,
    initial_step_size=0.02,
    jitter=0.02,
    seed=42,
)

In [ ]:
# Trace plots for OT
samples_z_ot = out_ot['samples_z']

fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
for i, ax in enumerate(axes.flat):
    for b in range(samples_z_ot.shape[1]):
        ax.plot(samples_z_ot[:, b, i], alpha=0.7, lw=0.8, label=f'chain {b}')
    ax.set_title(param_names[i])
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(loc='upper right', fontsize=8)
plt.suptitle('OT N=80 Trace Plots', fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Comparison Summary

In [ ]:
from advanced_particle_filter.hmc.run_hmc_poc import compute_rhat
from advanced_particle_filter.hmc.parameterization import (
    MU_START, MU_END, PHI_START, PHI_END, SIGMA_START, SIGMA_END,
)

mu_true = np.array([-1.0, 0.5])
Phi_true = np.array([[0.85, 0.12], [0.02, 0.90]])
Sigma_eta_true = np.array([[0.0225, 0.018], [0.018, 0.16]])

print('='*90)
print('  SVSSM HMC: Soft vs OT Comparison')
print('='*90)
print(f'{"Method":<22} {"N":>5} {"L":>3} {"Accept":>8} {"R-hat avg":>10} '
      f'{"mu":<20} {"Phi diag":<16} {"Sig diag":<16} {"Time":>7}')
print('-'*90)

for label, out, N, L in [
    ('Soft N=1000', out_soft, 1000, 15),
    ('OT N=80',     out_ot,  80,   5),
]:
    rhat = compute_rhat(tf.constant(out['samples_z'])).numpy()
    mu_mean = out['mus'].mean(0)
    phi_diag = [out['Phis'].mean(0)[0,0], out['Phis'].mean(0)[1,1]]
    sig_diag = [out['Sigmas'].mean(0)[0,0], out['Sigmas'].mean(0)[1,1]]
    
    print(f'{label:<22} {N:>5} {L:>3} {out["accept_rate"]:>7.1%} {rhat.mean():>10.1f} '
          f'({mu_mean[0]:+.2f}, {mu_mean[1]:+.2f})   '
          f'({phi_diag[0]:.3f}, {phi_diag[1]:.3f})  '
          f'({sig_diag[0]:.4f}, {sig_diag[1]:.4f})  '
          f'{out["elapsed"]:>6.0f}s')

print('-'*90)
print(f'{"Truth":<22} {"":>5} {"":>3} {"":>8} {"":>10} '
      f'({mu_true[0]:+.2f}, {mu_true[1]:+.2f})   '
      f'({Phi_true[0,0]:.3f}, {Phi_true[1,1]:.3f})  '
      f'({Sigma_eta_true[0,0]:.4f}, {Sigma_eta_true[1,1]:.4f})')

print(f'\nMemory comparison:')
print(f'  Soft N=1000: backward ~ {150*4*1000*2*8/1e6:.0f} MB')
print(f'  OT   N=80:   backward ~ {150*100*4*80**2*8/1e9:.1f} GB')
print(f'  Ratio: {150*100*4*80**2*8 / (150*4*1000*2*8):.0f}x more memory for {1000/80:.0f}x fewer particles')